# 🚦 Indian Traffic Signs — YOLO Fine-Tuning (Google Colab T4 GPU)

**Project:** AI-Powered Accessibility Assistant for Visually Impaired Students  
**CSE AIML Major Project — Project No. 40**

This notebook fine-tunes YOLO11n on the **Indian Traffic Signboards dataset**:  
- 7,833 images · 57 sign classes · COCO format  
- Classes include: stop, pedestrian crossing, school ahead, speed limits, hospital ahead, etc.

## Prerequisites
1. ✅ Enable T4 GPU: *Runtime → Change runtime type → T4 GPU*
2. ✅ Upload `archive (3).zip` to Google Drive at `MyDrive/AI_Accessibility_Assistant/`
3. ✅ Run all cells top to bottom

## Expected Results (T4 GPU, 80 epochs)
| Metric | Expected |
|--------|----------|
| mAP50 | 0.35 – 0.55 |
| mAP50-95 | 0.20 – 0.35 |
| Precision | 0.50 – 0.70 |
| Recall | 0.40 – 0.60 |
| Training time | ~45–75 min |

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install ultralytics>=8.1.0 --quiet

from ultralytics import YOLO
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")
ultralytics.checks()

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/AI_Accessibility_Assistant'
DRIVE_MODELS  = f'{DRIVE_PROJECT}/models'
DRIVE_RESULTS = f'{DRIVE_PROJECT}/training_results'

os.makedirs(DRIVE_MODELS,  exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f"Drive ready: {DRIVE_PROJECT}")

## 2. Dataset Setup

In [ ]:
# =============================================================
# OPTION D — Use archive (3).zip  ← DEFAULT (our dataset)
# Upload archive (3).zip to Google Drive:
#   MyDrive/AI_Accessibility_Assistant/archive (3).zip
# =============================================================
import os, json, yaml

DATASET_DIR = '/content/traffic_data'

zip_candidates = [
    f'{DRIVE_PROJECT}/archive (3).zip',
    '/content/archive (3).zip',
    '/content/archive(3).zip',
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if zip_path:
    print(f"Found zip: {zip_path}")
    os.makedirs(DATASET_DIR, exist_ok=True)
    !unzip -q "{zip_path}" -d "{DATASET_DIR}"
    print("Unzipped successfully.")
else:
    print("ZIP not found. Looking for already-extracted dataset...")

# Locate traffic_dataset
TRAFFIC_DATASET = None
for candidate in [
    f'{DATASET_DIR}/traffic_dataset',
    f'{DATASET_DIR}/archive (3)/traffic_dataset',
    '/content/traffic_dataset',
]:
    if os.path.isdir(candidate):
        TRAFFIC_DATASET = candidate
        break

if not TRAFFIC_DATASET:
    raise FileNotFoundError("traffic_dataset not found. Please upload archive (3).zip.")

print(f"Dataset path: {TRAFFIC_DATASET}")

# Count images per split
for split in ['train', 'valid', 'test']:
    n = len([f for f in os.listdir(f'{TRAFFIC_DATASET}/{split}') if f.endswith('.jpg')])
    print(f"  {split}: {n} images")

In [ ]:
# Build dataset YAML from COCO annotations (auto-extracts 57 class names)
YAML_PATH = '/content/traffic_signs.yaml'

coco_json = f'{TRAFFIC_DATASET}/train/_annotations.coco.json'
with open(coco_json) as f:
    coco = json.load(f)

cats = sorted(coco['categories'], key=lambda c: c['id'])
names = {i: c['name'] for i, c in enumerate(cats)}

cfg = {
    'path': TRAFFIC_DATASET,
    'train': 'train',
    'val':   'valid',
    'test':  'test',
    'nc':    len(names),
    'names': names,
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f)

print(f"Created {YAML_PATH} with {len(names)} classes")
print("Classes:", list(names.values())[:10], '...')

## 3. Training (80 epochs · T4 GPU · 640px)

In [ ]:
# Training hyperparameters
BASE_MODEL  = 'yolo11n.pt'   # Nano — small footprint, 57-class head
EPOCHS      = 80             # T4 GPU ~45–75 min
IMG_SIZE    = 640            # Full res for best accuracy
BATCH_SIZE  = 32             # T4 has 15GB; reduce to 16 if OOM
PATIENCE    = 20             # Early stopping
RUN_DIR     = '/content/runs'
RUN_NAME    = 'traffic_signs_v1'

print(f"Model:      {BASE_MODEL}")
print(f"Epochs:     {EPOCHS}")
print(f"Img size:   {IMG_SIZE}px")
print(f"Batch:      {BATCH_SIZE}")
print(f"Device:     {'GPU 0' if torch.cuda.is_available() else 'CPU (slow!)'}") 

In [ ]:
# Load model
model = YOLO(BASE_MODEL)
print(f"Loaded: {BASE_MODEL}")

In [ ]:
# START TRAINING
# T4 GPU ~45–75 min for 80 epochs on 7833 images
try:
    results = model.train(
        data=YAML_PATH,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        device=0 if torch.cuda.is_available() else 'cpu',
        project=RUN_DIR,
        name=RUN_NAME,
        exist_ok=True,
        save=True,
        save_period=10,
        plots=True,
        verbose=True,
        seed=42,
        # Optimizer
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        warmup_epochs=3,
        cos_lr=True,
        # Augmentation
        mosaic=1.0,
        close_mosaic=10,
        fliplr=0.5,
        degrees=5.0,
        translate=0.1,
        scale=0.4,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        mixup=0.1,
        # Loss
        box=7.5,
        cls=0.5,
        dfl=1.5,
    )
    print('\n✅ Training complete!')

except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print(f'⚠ OOM — retrying with batch={BATCH_SIZE // 2}')
        torch.cuda.empty_cache()
        results = model.train(
            data=YAML_PATH, epochs=EPOCHS, imgsz=IMG_SIZE,
            batch=BATCH_SIZE // 2, device=0, project=RUN_DIR,
            name=RUN_NAME, exist_ok=True, plots=True,
        )
    else:
        raise

## 4. Evaluation

In [ ]:
best_pt = f'{RUN_DIR}/{RUN_NAME}/weights/best.pt'

if os.path.exists(best_pt):
    best = YOLO(best_pt)
    test_metrics = best.val(
        data=YAML_PATH, split='test',
        conf=0.25, iou=0.45,
        device=0 if torch.cuda.is_available() else 'cpu',
        plots=True,
    )
    print('\n' + '='*50)
    print('TEST SET RESULTS')
    print('='*50)
    print(f'mAP@0.5:      {test_metrics.box.map50:.4f}')
    print(f'mAP@0.5:0.95: {test_metrics.box.map:.4f}')
    print(f'Precision:    {test_metrics.box.mp:.4f}')
    print(f'Recall:       {test_metrics.box.mr:.4f}')
    print('='*50)
else:
    print('❌ best.pt not found.')

In [ ]:
# Show training plots
from IPython.display import Image, display

plots_dir = f'{RUN_DIR}/{RUN_NAME}'
for plot in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    p = f'{plots_dir}/{plot}'
    if os.path.exists(p):
        print(f'\n{plot}:')
        display(Image(filename=p, width=700))

## 5. Live Inference on 5 Test Images

In [ ]:
test_dir = f'{TRAFFIC_DATASET}/test'
test_imgs = [f for f in os.listdir(test_dir) if f.endswith('.jpg')][:5]

if test_imgs and os.path.exists(best_pt):
    best = YOLO(best_pt)
    for img_name in test_imgs:
        img_path = f'{test_dir}/{img_name}'
        results = best.predict(img_path, conf=0.25, save=True,
                               project='/content/predictions', name='test')
        print(f'\n{img_name[:50]}:')
        for r in results:
            for box in r.boxes:
                cls  = int(box.cls[0])
                conf = float(box.conf[0])
                name = r.names[cls]
                print(f'  → {name}  conf={conf:.2%}')
else:
    print('No test images or model not found.')

## 6. Save to Google Drive

In [ ]:
import shutil
from datetime import datetime

weights_dir = f'{RUN_DIR}/{RUN_NAME}/weights'

if os.path.exists(weights_dir):
    for wf in ['best.pt', 'last.pt']:
        src = f'{weights_dir}/{wf}'
        if os.path.exists(src):
            dst = f'{DRIVE_MODELS}/{wf}'
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1e6
            print(f'✅ Saved {wf} → {dst}  ({size_mb:.1f} MB)')

    ts = datetime.now().strftime('%Y%m%d_%H%M')
    dst_run = f'{DRIVE_RESULTS}/run_{ts}'
    shutil.copytree(f'{RUN_DIR}/{RUN_NAME}', dst_run, dirs_exist_ok=True)
    print(f'✅ Training results → {dst_run}')

    info = {
        'date':       datetime.now().isoformat(),
        'base_model': BASE_MODEL,
        'epochs':     EPOCHS,
        'imgsz':      IMG_SIZE,
        'batch':      BATCH_SIZE,
        'nc':         57,
        'dataset':    'Indian Traffic Signboards (7833 images)',
        'gpu':        torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'map50':      float(test_metrics.box.map50) if 'test_metrics' in dir() else None,
    }
    with open(f'{DRIVE_MODELS}/training_info.json', 'w') as f:
        json.dump(info, f, indent=2)

    print('\n' + '='*50)
    print('NEXT STEPS:')
    print(f'  1. Download best.pt from: {DRIVE_MODELS}/best.pt')
    print('  2. Copy to project:        models/custom/best.pt')
    print('  3. Backend auto-loads it on restart.')
    print('='*50)
else:
    print('❌ No weights found.')

In [ ]:
# Optional: download best.pt directly to your computer
if os.path.exists(f'{weights_dir}/best.pt'):
    from google.colab import files
    files.download(f'{weights_dir}/best.pt')
    print('Download started. Place it at models/custom/best.pt in your project.')